## Find the file path

In [0]:
display(
    dbutils.fs.ls(
        "abfss://bronze@adventureworkstorageadls.dfs.core.windows.net/AdventureWorks2025/"
    )
)

## Read CSV Filse

In [0]:
df_customer = spark.read.option("header", "true").option("inferSchema", "true").csv(
    "abfss://bronze@adventureworkstorageadls.dfs.core.windows.net/AdventureWorks2025/DimCustomer.csv")
    
display(df_customer.limit(5))

In [0]:
df_product = spark.read.option("header", "true").option("inferSchema", "true").csv(
    "abfss://bronze@adventureworkstorageadls.dfs.core.windows.net/AdventureWorks2025/DimProduct.csv"
)

df_date = spark.read.option("header", "true").option("inferSchema", "true").csv(
    "abfss://bronze@adventureworkstorageadls.dfs.core.windows.net/AdventureWorks2025/DimDate.csv"
)

df_geography = spark.read.option("header", "true").option("inferSchema", "true").csv(
    "abfss://bronze@adventureworkstorageadls.dfs.core.windows.net/AdventureWorks2025/DimGeography.csv"
)

df_sales = spark.read.option("header", "true").option("inferSchema", "true").csv(
    "abfss://bronze@adventureworkstorageadls.dfs.core.windows.net/AdventureWorks2025/FactInternetSales.csv"
)

print("All SQL Bronze DataFrames loaded successfully")


## Read API Bronze data.

In [0]:
df_api = spark.read.option("multiline", "true").json("abfss://bronze@adventureworkstorageadls.dfs.core.windows.net/API/frankfurter_usd_inr.json")

display(df_api)

## Clean & transform SQL data

In [0]:
from pyspark.sql.functions import col, trim

dm_customer_clean = df_customer.select(
                            *[trim(col(c)).alias(c) for c in df_customer.columns])

                           
display(dm_customer_clean.limit(5))

## Make dynamic approach

In [0]:
from pyspark.sql.functions import col, trim

def clean_dataframe(df):
    return df.select(
        *[
            trim(col(c)).alias(c)
            for c in df.columns
        ]
    )
sql_dataframes = {
    "DimCustomer": df_customer,
    "DimProduct": df_product,
    "DimDate": df_date,
    "DimGeography": df_geography,
    "FactInternetSales": df_sales
}

cleaned_dataframes = {name: clean_dataframe(df) for name, df in sql_dataframes.items()}

df_customer_clean = cleaned_dataframes["DimCustomer"]
df_product_clean = cleaned_dataframes["DimProduct"]
df_date_clean = cleaned_dataframes["DimDate"]
df_geography_clean = cleaned_dataframes["DimGeography"]
df_sales_clean = cleaned_dataframes["FactInternetSales"]

## Clean & transform API data

In [0]:
from pyspark.sql.functions import col

df_api_clean = df_api.select(
    col("amount").cast("double").alias("amount"),
    col("base").alias("base_currency"),
    col("date").alias("exchange_date"),
    col("rates.INR").cast("double").alias("INR_rate")
)

display(df_api_clean)

## Handling Nulls

## Silver data-quality transformation

1.Trim string columns.

2.Preserve NULL values.

3.Keep numeric/date NULLs as NULL.

4.Remove completely empty rows if any.

In [0]:
def clean_dataframe(df):
    # Trim whitespace from string columns while preserving NULLs
    for field in df.schema.fields:
        if field.dataType.simpleString() == "string":
            df = df.withColumn(field.name, trim(col(field.name)))

    # Remove completely empty rows
    df = df.dropna(how="all")

    return df


cleaned_dataframes = {
    name: clean_dataframe(df)
    for name, df in sql_dataframes.items()
}

print("Silver transformations completed for all SQL tables.")

## metadata-driven type-mapping layer.

In [0]:
type_mapping = {
    "DimCustomer": {
        "CustomerKey": "int",
        "GeographyKey": "int",
        "NameStyle": "boolean",
        "HouseOwnerFlag": "boolean",
        "NumberCarsOwned": "int",
        "NumberChildrenAtHome": "int",
        "TotalChildren": "int",
        "YearlyIncome": "decimal(18,2)",
        "DateFirstPurchase": "date",
        "BirthDate": "date"
    },

    "DimDate": {
        "DateKey": "int",
        "CalendarQuarter": "int",
        "CalendarSemester": "int",
        "CalendarYear": "int",
        "DayNumberOfMonth": "int",
        "DayNumberOfWeek": "int",
        "DayNumberOfYear": "int",
        "FiscalQuarter": "int",
        "FiscalSemester": "int",
        "FiscalYear": "int",
        "MonthNumberOfYear": "int",
        "WeekNumberOfYear": "int",
        "FullDateAlternateKey": "date"
    },

    "DimGeography": {
        "GeographyKey": "int",
        "SalesTerritoryKey": "int"
    },

    "FactInternetSales": {
        "CurrencyKey": "int",
        "CustomerKey": "int",
        "DueDateKey": "int",
        "OrderDateKey": "int",
        "ProductKey": "int",
        "PromotionKey": "int",
        "RevisionNumber": "int",
        "SalesOrderLineNumber": "int",
        "SalesTerritoryKey": "int",
        "ShipDateKey": "int",
        "OrderQuantity": "int",
        "DiscountAmount": "decimal(18,2)",
        "ExtendedAmount": "decimal(18,2)",
        "Freight": "decimal(18,2)",
        "ProductStandardCost": "decimal(18,2)",
        "SalesAmount": "decimal(18,2)",
        "TaxAmt": "decimal(18,2)",
        "UnitPrice": "decimal(18,2)",
        "UnitPriceDiscountPct": "decimal(18,4)"
    }
}


In [0]:
def apply_type_mapping(df, table_name):
    mappings = type_mapping.get(table_name, {})

    for column_name, data_type in mappings.items():
        if column_name in df.columns:
            df = df.withColumn(
                column_name,
                col(column_name).cast(data_type)
            )

    return df

In [0]:
cleaned_dataframes = {
    table_name: apply_type_mapping(df, table_name)
    for table_name, df in sql_dataframes.items()
}

for table_name, df in cleaned_dataframes.items():
    print(f"\n--- {table_name} ---")
    df.printSchema()

In [0]:
type_mapping["DimProduct"] = {
    "ProductKey": "int",
    "ProductSubcategoryKey": "int",
    "StandardCost": "decimal(18,2)",
    "FinishedGoodsFlag": "boolean",
    "SafetyStockLevel": "int",
    "ReorderPoint": "int",
    "ListPrice": "decimal(18,2)",
    "Weight": "decimal(18,2)",
    "DaysToManufacture": "int",
    "DealerPrice": "decimal(18,2)",
    "StartDate": "date",
    "EndDate": "date"
}

print("DimProduct type mapping added")

In [0]:
type_mapping["DimProduct"].update({
    "StandardCost": "decimal(18,2)",
    "ListPrice": "decimal(18,2)",
    "DealerPrice": "decimal(18,2)",
    "Weight": "decimal(18,2)",
    "StartDate": "date",
    "EndDate": "date"
})

type_mapping["FactInternetSales"].update({
    "TotalProductCost": "decimal(18,2)",
    "OrderDate": "date",
    "DueDate": "date",
    "ShipDate": "date"
})

In [0]:
silver_dataframes = {
    table_name: apply_type_mapping(df, table_name)
    for table_name, df in sql_dataframes.items()
}

for table_name, df in silver_dataframes.items():
    print(f"\n--- {table_name} ---")
    df.printSchema()

## Transform DimCustomer Table

In [0]:
df_customer_silver = silver_dataframes["DimCustomer"]

display(
    df_customer_silver.select(
        *[
            df_customer_silver[c].isNull().alias(c)
            for c in df_customer_silver.columns
        ]
    ).limit(10)
)

In [0]:
from pyspark.sql.functions import col, coalesce, lit, concat_ws, trim, when

df_customer_clean = (
    silver_dataframes["DimCustomer"]
    .drop("Title", "Suffix", "AddressLine2")
    .withColumn(
        "MiddleName",
        coalesce(col("MiddleName"), lit(""))
    )
    .withColumn(
        "FullName",
        trim(
            concat_ws(
                " ",
                trim(col("FirstName")),
                trim(col("MiddleName")),
                trim(col("LastName"))
            )
        )
    )
    .withColumn(
        "MaritalStatus",
        when(col("MaritalStatus") == "M", "Married")
        .when(col("MaritalStatus") == "S", "Single")
        .otherwise(col("MaritalStatus"))
    )
    .withColumn(
        "Gender",
        when(col("Gender") == "M", "Male")
        .when(col("Gender") == "F", "Female")
        .otherwise(col("Gender"))
    )
    .drop("FirstName", "MiddleName", "LastName")
)

display(df_customer_clean.limit(10))

In [0]:
from pyspark.sql.functions import col, sum, when

null_summary = df_customer_clean.select(
    *[
        sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
        for c in df_customer_clean.columns
    ]
)

display(null_summary)

## Transform DimProduct Tabale

In [0]:
display(
    silver_dataframes["DimProduct"].limit(10)
)

In [0]:
from pyspark.sql.functions import col, sum, when

df_product_silver = silver_dataframes["DimProduct"]

null_summary_product = df_product_silver.select(
    *[
        sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
        for c in df_product_silver.columns
    ]
)

display(null_summary_product.limit(10))

In [0]:
display(
    df_product_silver
    .groupBy("Status")
    .count()
    .orderBy("Status")
)

In [0]:
from pyspark.sql.functions import col, coalesce, lit

df_product_clean = (
    df_product_silver

    # Replace missing categorical values
    .withColumn(
        "WeightUnitMeasureCode",
        coalesce(col("WeightUnitMeasureCode"), lit("N/A"))
    )
    .withColumn(
        "SizeUnitMeasureCode",
        coalesce(col("SizeUnitMeasureCode"), lit("N/A"))
    )
    .withColumn(
        "Size",
        coalesce(col("Size"), lit("N/A"))
    )
    .withColumn(
        "ProductLine",
        coalesce(col("ProductLine"), lit("N/A"))
    )
    .withColumn(
        "Class",
        coalesce(col("Class"), lit("N/A"))
    )
    .withColumn(
        "Style",
        coalesce(col("Style"), lit("N/A"))
    )
    .withColumn(
        "ModelName",
        coalesce(col("ModelName"), lit("N/A"))
    )
    .withColumn(
        "Status",
        coalesce(col("Status"), lit("N/A"))
    )

    # Remove unnecessary columns
    .drop(
        "SpanishProductName",
        "FrenchProductName",
        "LargePhoto",
        "EnglishDescription",
        "FrenchDescription",
        "ChineseDescription",
        "ArabicDescription",
        "HebrewDescription",
        "ThaiDescription",
        "GermanDescription",
        "JapaneseDescription",
        "TurkishDescription"
    )
)

display(df_product_clean.limit(20))

In [0]:
df_date_clean = (
    silver_dataframes["DimDate"]
)

display(df_date_clean)

In [0]:
from pyspark.sql.functions import col, sum

display(
    df_date_clean.select(
        [
            sum(col(c).isNull().cast("int")).alias(c)
            for c in df_date_clean.columns
        ]
    )
)

In [0]:
df_geography_clean = (
    silver_dataframes["DimGeography"]
)

display(df_geography_clean.limit(10))

In [0]:
from pyspark.sql.functions import col, sum

display(
    df_geography_clean.select(
        [
            sum(col(c).isNull().cast("int")).alias(c)
            for c in df_geography_clean.columns
        ]
    )
)

In [0]:
display(
    df_geography_clean.select(
        "GeographyKey",
        "City",
        "StateProvinceCode",
        "StateProvinceName",
        "CountryRegionCode",
        "EnglishCountryRegionName",
        "PostalCode",
        "SalesTerritoryKey"
    ).limit(10)
)

In [0]:
from pyspark.sql.functions import col, sum

df_fact_clean = (
    silver_dataframes["FactInternetSales"]
)

display(df_fact_clean)

In [0]:
display(
    df_fact_clean.select(
        [
            sum(col(c).isNull().cast("int")).alias(c)
            for c in df_fact_clean.columns
        ]
    )
)

In [0]:
display(
    df_fact_clean.select(
        "SalesOrderNumber",
        "OrderDate",
        "DueDate",
        "ShipDate",
        "CustomerKey",
        "ProductKey",
        "OrderQuantity",
        "UnitPrice",
        "DiscountAmount",
        "SalesAmount",
        "TaxAmt",
        "Freight",
        "TotalProductCost"
    )
)

In [0]:
display(
    df_fact_clean.filter(
        col("SalesAmount") < 0
    )
)

In [0]:
print("FactInternetSales rows:", df_fact_clean.count())

# write files

In [0]:
silver_base = "abfss://silver@adventureworkstorageadls.dfs.core.windows.net/AdventureWorks2025"

silver_final = {
    "DimCustomer": df_customer_clean,
    "DimProduct": df_product_clean,
    "DimDate": df_date_clean,
    "DimGeography": df_geography_clean,
    "FactInternetSales": df_fact_clean
}

for table_name, df in silver_final.items():
    (
        df.write
        .mode("overwrite")
        .parquet(f"{silver_base}/{table_name}")
    )

print("All finalized SQL Silver tables written successfully.")

In [0]:
for table_name, df in silver_final.items():
    print(f"{table_name}: {df.count()} rows")

In [0]:
display(
    df_api_clean
)

In [0]:

print("API Silver rows:", df_api_clean.count())

In [0]:
silver_base = "abfss://silver@adventureworkstorageadls.dfs.core.windows.net/AdventureWorks2025"

df_customer_check = spark.read.parquet(
    f"{silver_base}/DimCustomer"
)

print("Rows:", df_customer_check.count())
print("Columns:")
print(df_customer_check.columns)

display(df_customer_check.limit(10))